In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import levy_stable
from sklearn.impute import SimpleImputer
print("''")

''


In [2]:


# Load the dataset
df_updated = pd.read_csv('Updated_Imputed_Dataset.csv')  # Path to the uploaded dataset

C:\Users\dinit\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3457: DtypeWarning: Columns (17,25,26,27,29,30,32,34,35,36,37,38,41,43,44,45,46,47,48,49,50,51,52) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [3]:


# Display the first few rows of the dataset to understand its structure
print(df_updated.head())

# Handle mixed data types: Ensure all relevant columns are numeric where necessary
# Convert columns with numeric-like data to numeric, forcing errors to NaN
numerical_columns = df_updated.select_dtypes(include=['float64', 'int64']).columns
df_updated[numerical_columns] = df_updated[numerical_columns].apply(pd.to_numeric, errors='coerce')

# For categorical columns (object type), convert them to the most frequent value
categorical_columns = df_updated.select_dtypes(include=['object']).columns

# Impute missing values in numerical columns with 'mean'
numerical_imputer = SimpleImputer(strategy='mean')
df_updated[numerical_columns] = numerical_imputer.fit_transform(df_updated[numerical_columns])

# Impute missing values in categorical columns with 'most_frequent' (mode)
categorical_imputer = SimpleImputer(strategy='most_frequent')
df_updated[categorical_columns] = categorical_imputer.fit_transform(df_updated[categorical_columns])

# Check if there are any missing values left after imputation
missing_values_after_imputation = df_updated.isnull().sum()

# Display the result to confirm there are no missing values left
print(f"Missing values after imputation:\n{missing_values_after_imputation}")

# Save the cleaned and imputed dataset to a new CSV file
output_file_path = 'Cleaned_Updated_Imputed_Dataset.csv'  # Replace with your desired file path
df_updated.to_csv(output_file_path, index=False)

print(f"Dataset saved to: {output_file_path}")


   Unnamed: 0  Record ID Agency Code  Agency Name       Agency Type  \
0           0        3.0     AK00101    Anchorage  Municipal Police   
1           1        5.0     AK00101    Anchorage  Municipal Police   
2           2      650.0     AR04700  Mississippi           Sheriff   
3           3      945.0     AZ00723      Phoenix  Municipal Police   
4           4      979.0     AZ01000         Pima           Sheriff   

          City     State    Year     Month  Incident  ...  \
0    Anchorage    Alaska  1980.0     March       2.0  ...   
1    Anchorage    Alaska  1980.0     April       2.0  ...   
2  Mississippi  Arkansas  1980.0  December       1.0  ...   
3     Maricopa   Arizona  1980.0  December       9.0  ...   
4         Pima   Arizona  1980.0   October       2.0  ...   

  Victim Education and Family Background Victim If Randomly Chosen or Planned  \
0                                    NaN                                  NaN   
1                                    NaN    

In [ ]:
# Extract the relevant data column for Mogul distribution fitting (e.g., "Offender No. of Homicides")
homicides_data = df_updated['Offender No. of Homicides']

# Clean data: Remove NaN, Inf, or -Inf values (if any remaining after imputation)
homicides_data_clean = homicides_data.replace([np.inf, -np.inf], np.nan).dropna()

# Check if any non-finite values are left
print(f"Non-finite values in the cleaned data: {homicides_data_clean.isnull().sum()}")

homicides_data_clean = pd.to_numeric(homicides_data, errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()

# Fit the stable distribution (approximating the Mogul distribution)
#params_stable = levy_stable.fit(homicides_data_clean)
#params_stable = levy_stable.fit(homicides_data_clean, alpha=1.5, beta=0, loc=homicides_data_clean.mean(), scale=homicides_data_clean.std()) # Provide initial guesses
params_stable = levy_stable.fit(homicides_data_clean, 1.5, 0, loc=homicides_data_clean.mean(), scale=homicides_data_clean.std())  # Provide initial guesses
print("Fit the stable distribution with a reduced number of iterations and tolerance")
#params_stable = levy_stable.fit(homicides_data_clean, maxiter=100, tol=1e-5)



# Generate values for the fitted stable distribution
x_values = np.linspace(min(homicides_data_clean), max(homicides_data_clean), 100)
stable_pdf = levy_stable.pdf(x_values, *params_stable)

# Plot the actual data and the fitted stable distribution
plt.figure(figsize=(8, 6))
plt.hist(homicides_data_clean, bins=np.arange(0, int(max(homicides_data_clean)) + 2) - 0.5, density=True, alpha=0.6, label="Actual Data", color='blue')
plt.plot(x_values, stable_pdf, 'r-', lw=2, label="Fitted Stable Distribution (Mogul-like)")
plt.title('Stable Distribution Fit for Offender Homicides')
plt.xlabel('Number of Homicides')
plt.ylabel('Density')
plt.legend()
plt.show()


Non-finite values in the cleaned data: 0
